In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đọc dữ liệu đã tiền xử lý
df = pd.read_csv("../Data/processed/appliances_processed.csv")

# Tách đặc trưng và biến mục tiêu
X = df.drop(columns=["Appliances", "date"])
y = df["Appliances"]

print("Kích thước dữ liệu:", df.shape)
print("Số đặc trưng:", X.shape[1])
print("Số dòng:", len(df))
print("Dữ liệu thiếu:", df.isnull().sum().sum())

Kích thước dữ liệu: (19732, 34)
Số đặc trưng: 32
Số dòng: 19732
Dữ liệu thiếu: 0


In [3]:
# Các tỷ lệ chia dữ liệu cần thử nghiệm
train_ratios = [0.7, 0.8, 0.9]

# Lưu kết quả
results = []

for ratio in train_ratios:

    # Chia dữ liệu theo thứ tự thời gian
    split_index = int(len(df) * ratio)

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    # Sử dụng Linear Regression
    model = LinearRegression()

    # Huấn luyện mô hình
    model.fit(X_train, y_train)

    # Dự đoán
    y_pred = model.predict(X_test)

    # Đánh giá
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Train/Test": f"{round(ratio*100)}/{100-round(ratio*100)}",
        "Train samples": len(X_train),
        "Test samples": len(X_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# Hiển thị bảng kết quả
experiment_df = pd.DataFrame(results)

print("BẢNG KẾT QUẢ THỰC NGHIỆM")
display(experiment_df.round(4))

BẢNG KẾT QUẢ THỰC NGHIỆM


,Train/Test,Train samples,Test samples,MAE,RMSE,R2
0,70/30,13812,5920,29.6113,62.2602,0.5378
1,80/20,15785,3947,29.0102,62.0053,0.5361
2,90/10,17758,1974,27.9618,62.7410,0.5537


In [4]:
models = {
    "Baseline": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

train_ratios = [0.7, 0.8, 0.9]
comparison_results = []

for ratio in train_ratios:

    split_index = int(len(df) * ratio)

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    for model_name, model in models.items():

        print(f"Đang chạy: {model_name} - {int(ratio*100)}/{100-int(ratio*100)}")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        comparison_results.append({
            "Train/Test": f"{int(ratio*100)}/{100-int(ratio*100)}",
            "Model": model_name,
            "MAE": mean_absolute_error(y_test, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "R2": r2_score(y_test, y_pred)
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.round(4))

Đang chạy: Baseline - 70/30
Đang chạy: Linear Regression - 70/30
Đang chạy: Ridge - 70/30
Đang chạy: Random Forest - 70/30
Đang chạy: Gradient Boosting - 70/30
Đang chạy: Baseline - 80/20
Đang chạy: Linear Regression - 80/20
Đang chạy: Ridge - 80/20
Đang chạy: Random Forest - 80/20
Đang chạy: Gradient Boosting - 80/20
Đang chạy: Baseline - 90/10
Đang chạy: Linear Regression - 90/10
Đang chạy: Ridge - 90/10
Đang chạy: Random Forest - 90/10
Đang chạy: Gradient Boosting - 90/10


,Train/Test,Model,MAE,RMSE,R2
0,70/30,Baseline,53.4086,91.6539,-0.0016
1,70/30,Linear Regression,29.6113,62.2602,0.5378
2,70/30,Ridge,29.6159,62.2609,0.5378
3,70/30,Random Forest,64.7515,111.0613,-0.4706
4,70/30,Gradient Boosting,106.9155,153.9727,-1.8266
5,80/20,Baseline,52.6886,91.0536,-0.0003
6,80/20,Linear Regression,29.0102,62.0053,0.5361
7,80/20,Ridge,29.0117,62.0055,0.5361
8,80/20,Random Forest,77.3325,114.3738,-0.5783
9,80/20,Gradient Boosting,110.9029,144.9163,-1.5339


In [6]:
comparison_df.to_csv(
    "../reports/experiment_comparison.csv",
    index=False
)

print("Đã lưu kết quả thực nghiệm!")

Đã lưu kết quả thực nghiệm!
